# UR5e pi0-base LoRA Fine-Tuning

> **TODO / WIP:** This notebook is a work-in-progress template for pi0-base training. It has not been run yet — `pi0_ur5_base_v1` has no trained checkpoint. The pi0-FAST variant (`train_pi0fast_ur5_colab_032226.ipynb`) is the active training notebook.

**All cells run in Google Colab Pro (A100 recommended) unless marked `[LAPTOP]`.**

## Requirements
- Google Colab Pro with A100 GPU (Runtime → Change runtime type → A100)
- HuggingFace account with a **write** token (`sheilsarda/pi0_ur5_base_v1`)
- W&B account (wandb.ai)
- Dataset already on HuggingFace Hub: `sheilsarda/ur5_isaac_sim_v1`

## Current state
- `pi0_ur5_base_v1`: not yet trained

In [ ]:
# Cell 1: Verify GPU
!nvidia-smi

In [ ]:
# Cell 2: Clone repo and install dependencies
!git clone https://github.com/sheilsarda/openpi.git
%cd openpi
!pip install uv -q
!uv sync

In [ ]:
# Cell 3: Authenticate HuggingFace (needed to download dataset AND upload checkpoints)
from huggingface_hub import login

# IMPORTANT: You must use a WRITE token to upload checkpoints later.
# Get your token from: https://huggingface.co/settings/tokens
login()

In [ ]:
# Cell 4: Authenticate W&B
import wandb
wandb.login()  # paste your W&B API key

In [ ]:
# Cell 5: Authenticate Google Cloud
# Required to download pretrained weights from gs://openpi-assets/
from google.colab import auth
auth.authenticate_user()

In [ ]:
# Cell 6: Mount Google Drive — checkpoints will be saved here
# so they survive the Colab session ending.
# NOTE: Run the clone cell (Cell 2) before this one.
from google.colab import drive
import os

drive.mount('/content/drive')

os.makedirs('/content/drive/MyDrive/openpi_checkpoints', exist_ok=True)

# Symlink so openpi writes checkpoints directly to Drive.
# os.symlink(src, dst): src=real path on Drive, dst=path inside the repo.
if not os.path.islink('/content/openpi/checkpoints') and not os.path.exists('/content/openpi/checkpoints'):
    os.symlink('/content/drive/MyDrive/openpi_checkpoints', '/content/openpi/checkpoints')

print('Checkpoints dir:', os.path.realpath('/content/openpi/checkpoints'))

---
## Training

Three cells below:
1. **Norm stats** — run once (or after any dataset change); fast, CPU-only
2. **Train: FROM SCRATCH** — wipes existing checkpoints, starts from step 0
3. **Train: RESUME** — continues from the latest checkpoint; `--num-train-steps` is the new total

Run **one** of cells 2 or 3, not both.

---
### pi0-base (`pi0_ur5_base`)

In [ ]:
# Norm stats — pi0-base
# Skip if already computed and the dataset hasn't changed.
!uv run scripts/compute_norm_stats.py --config-name=pi0_ur5_base

In [ ]:
# TODO: Not yet run — pi0-base training has not been attempted.
# Train pi0-base — FROM SCRATCH
# Wipes checkpoints/pi0_ur5_base/ur5_base_v1/ and starts from step 0.
# W&B: creates a new run named ur5_base_v1.
!uv run scripts/train.py pi0_ur5_base \
  --exp-name ur5_base_v1 \
  --overwrite \
  --num-train-steps 10000

In [ ]:
# Train pi0-base — RESUME
# Continues from the latest checkpoint in checkpoints/pi0_ur5_base/ur5_base_v1/.
# --num-train-steps is the TOTAL target (e.g. already at 10k → set 20k to run 10k more).
# W&B: resumes the existing ur5_base_v1 run automatically.
!uv run scripts/train.py pi0_ur5_base \
  --exp-name ur5_base_v1 \
  --resume \
  --num-train-steps 20000

In [ ]:
# Upload pi0-base checkpoint to HuggingFace
# Uploads the full checkpoints/pi0_ur5_base/ur5_base_v1/ folder (all saved steps).
from huggingface_hub import HfApi
import os

api = HfApi()
repo_id = 'sheilsarda/pi0_ur5_base_v1'
api.create_repo(repo_id=repo_id, repo_type='model', exist_ok=True)

# Infer the latest step from the checkpoint directory for the commit message.
ckpt_dir = '/content/drive/MyDrive/openpi_checkpoints/pi0_ur5_base/ur5_base_v1'
steps = sorted(int(d) for d in os.listdir(ckpt_dir) if d.isdigit())
latest_step = steps[-1] if steps else '?'

api.upload_folder(
    folder_path=ckpt_dir,
    repo_id=repo_id,
    repo_type='model',
    commit_message=f'pi0-base LoRA ur5 checkpoint step {latest_step}',
)
print(f'Uploaded step {latest_step} → https://huggingface.co/{repo_id}')

---
## [LAPTOP] Download checkpoint for local inference

Run this cell **on your laptop** to pull a checkpoint from HuggingFace and serve it locally with openpi.

```bash
# Then serve:
cd ~/Development/openpi
uv run scripts/serve_policy.py policy:checkpoint \
    --policy.config=pi0_ur5_base \
    --policy.dir=checkpoints/pi0_ur5_base/ur5_base_v1/10000
```

In [ ]:
# [LAPTOP] Download pi0-base checkpoint from HuggingFace for local serving
#
# Prerequisites (run once):
#   cd ~/Development/openpi && source .venv/bin/activate
#   pip install huggingface_hub
#   huggingface-cli login   (read token is sufficient)
#
# This downloads all step subdirs (e.g. 1000/, 2000/, ..., 10000/) into the
# local checkpoints folder so serve_policy.py can find them.

from huggingface_hub import snapshot_download

local_dir = '/home/sheil/Development/openpi/checkpoints/pi0_ur5_base/ur5_base_v1'

path = snapshot_download(
    repo_id='sheilsarda/pi0_ur5_base_v1',
    repo_type='model',
    local_dir=local_dir,
)
print(f'Downloaded to: {path}')
print()
print('To serve step 10000:')
print('  cd ~/Development/openpi')
print('  uv run scripts/serve_policy.py policy:checkpoint \\')
print('      --policy.config=pi0_ur5_base \\')
print('      --policy.dir=checkpoints/pi0_ur5_base/ur5_base_v1/10000')